**Recall the libraries**

In [1]:
!pip install datasets
!pip install nltk scikit-learn
!pip install scikit-learn
!pip install tensorflow  # For the MLP model


**Load the dataset**

In [2]:
from datasets import load_dataset

In [3]:
# Load the CLINC Out-of-Scope dataset
dataset = load_dataset("clinc_oos", "imbalanced")

# Check the available splits
# print(dataset)

# Example data point
dataset['train'][:5]

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


{'text': ['what are the steps for setting up direct deposit for my paycheck',
  'how is a direct deposit set up',
  'how would i go about setting up a direct deposit',
  'tell me how to set up a direct deposit',
  'how do i arrange a direct deposit into my savings account'],
 'intent': [108, 108, 108, 108, 108]}

**Download NLTK resources**

In [4]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [5]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

**Preprocess the input text**

In [6]:
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Preprocessing function
def preprocess(text):
    text = text.lower()  # Lowercase
    text = re.sub(r'[^a-z\s]', '', text)  # Remove special chars
    tokens = word_tokenize(text)  # Tokenize
    tokens = [lemmatizer.lemmatize(token) for token in tokens if token not in stop_words]  # Remove stopwords + lemmatize
    return ' '.join(tokens)  # Return cleaned string


**Preprocess training data and extract labels**

In [7]:
# Apply to training data
texts = [preprocess(example['text']) for example in dataset['train']]
labels = [example['intent'] for example in dataset['train']]


**Convert text data into numerical features using TF-IDF vectorization**

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(texts)


**Encode the labels into numerical values using LabelEncoder**

In [9]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(labels)


**Split the data into training and testing sets**

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


**Train a Logistic Regression model and evaluate its performance**

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

print("Logistic Regression Results:")
# Convert label_encoder.classes_ to a list of strings
target_names = [str(cls) for cls in label_encoder.classes_]
print(classification_report(y_test, y_pred_lr, target_names=target_names))

Logistic Regression Results:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        15
           1       1.00      1.00      1.00        13
           2       0.75      1.00      0.86         3
           3       0.82      0.78      0.80        18
           4       0.53      0.83      0.65        12
           5       0.67      0.33      0.44         6
           6       0.25      1.00      0.40         1
           7       1.00      1.00      1.00         5
           8       0.92      0.96      0.94        25
           9       0.73      0.92      0.81        12
          10       0.92      0.89      0.91        27
          11       0.66      0.84      0.74        25
          12       1.00      0.62      0.77         8
          13       0.93      0.87      0.90        30
          14       1.00      0.83      0.91        12
          15       1.00      0.77      0.87        13
          16       0.86      0.79      0.83        2

**Train a Naïve Bayes model and evaluate its performance**

In [12]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

print("Naïve Bayes Results:")
# Convert label_encoder.classes_ to a list of strings
target_names = [str(cls) for cls in label_encoder.classes_]
print(classification_report(y_test, y_pred_nb, target_names=target_names)) # Pass the list of string labels

Naïve Bayes Results:
              precision    recall  f1-score   support

           0       1.00      0.93      0.97        15
           1       1.00      1.00      1.00        13
           2       1.00      0.33      0.50         3
           3       0.60      0.83      0.70        18
           4       0.25      0.08      0.12        12
           5       0.50      0.17      0.25         6
           6       0.00      0.00      0.00         1
           7       1.00      0.20      0.33         5
           8       0.81      1.00      0.89        25
           9       1.00      0.83      0.91        12
          10       0.91      0.74      0.82        27
          11       0.63      0.88      0.73        25
          12       1.00      0.12      0.22         8
          13       0.97      0.93      0.95        30
          14       0.36      0.83      0.50        12
          15       1.00      0.85      0.92        13
          16       0.86      0.75      0.80        24
      

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


**Train a Random Forest model and evaluate its performance**

In [13]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("Random Forest Results:")
# Convert label_encoder.classes_ to a list of strings
target_names = [str(cls) for cls in label_encoder.classes_]
print(classification_report(y_test, y_pred_rf, target_names=target_names)) # Pass the list of string labels

Random Forest Results:
              precision    recall  f1-score   support

           0       0.94      1.00      0.97        15
           1       1.00      1.00      1.00        13
           2       0.43      1.00      0.60         3
           3       0.76      0.89      0.82        18
           4       0.71      1.00      0.83        12
           5       0.75      0.50      0.60         6
           6       1.00      1.00      1.00         1
           7       1.00      1.00      1.00         5
           8       0.82      0.92      0.87        25
           9       0.86      1.00      0.92        12
          10       0.86      0.93      0.89        27
          11       0.64      0.64      0.64        25
          12       1.00      0.38      0.55         8
          13       0.83      0.83      0.83        30
          14       1.00      0.83      0.91        12
          15       1.00      0.92      0.96        13
          16       0.94      0.67      0.78        24
    

**Train a Multi-Layer Perceptron (MLP) neural network and evaluate its performance**

In [14]:
import tensorflow as tf
from tensorflow.keras import layers

mlp = tf.keras.Sequential([
    layers.Input(shape=(X.shape[1],)),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(151, activation='softmax')  # Changed to 151 classes to accommodate labels 0-150
])

mlp.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
mlp.fit(X_train.toarray(), y_train, epochs=5, batch_size=32, validation_split=0.1)

# Evaluation
y_pred_mlp = mlp.predict(X_test.toarray())
y_pred_mlp_classes = y_pred_mlp.argmax(axis=1)

from sklearn.metrics import classification_report
print("MLP Neural Network Results:")
# Convert label_encoder.classes_ to a list of strings
target_names = [str(cls) for cls in label_encoder.classes_]
print(classification_report(y_test, y_pred_mlp_classes, target_names=target_names)) # Pass the list of string labels

Epoch 1/5
240/240 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.2138 - loss: 4.8156 - val_accuracy: 0.5976 - val_loss: 3.6597
Epoch 2/5
240/240 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - accuracy: 0.6749 - loss: 3.0213 - val_accuracy: 0.7753 - val_loss: 1.7724
Epoch 3/5
240/240 ━━━━━━━━━━━━━━━━━━━━ 8s 26ms/step - accuracy: 0.8339 - loss: 1.3579 - val_accuracy: 0.8424 - val_loss: 1.0366
Epoch 4/5
240/240 ━━━━━━━━━━━━━━━━━━━━ 8s 33ms/step - accuracy: 0.9015 - loss: 0.7250 - val_accuracy: 0.8600 - val_loss: 0.7550
Epoch 5/5
240/240 ━━━━━━━━━━━━━━━━━━━━ 6s 26ms/step - accuracy: 0.9338 - loss: 0.4496 - val_accuracy: 0.8765 - val_loss: 0.6242
67/67 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step
MLP Neural Network Results:
              precision    recall  f1-score   support

           0       0.94      1.00      0.97        15
           1       1.00      1.00      1.00        13
           2       0.75      1.00      0.86         3
           3       0.83      0.83      0.83        18
           4       0.

**Compare the accuracy of different models**

In [15]:
from sklearn.metrics import accuracy_score

print("LR Accuracy:", accuracy_score(y_test, y_pred_lr))
print("NB Accuracy:", accuracy_score(y_test, y_pred_nb))
print("RF Accuracy:", accuracy_score(y_test, y_pred_rf))
print("MLP Accuracy:", accuracy_score(y_test, y_pred_mlp_classes))


LR Accuracy: 0.851764705882353
NB Accuracy: 0.7934117647058824
RF Accuracy: 0.8390588235294117
MLP Accuracy: 0.8672941176470588


**Extract the label names from the dataset**

In [16]:
label_names = dataset['train'].features['intent'].names



**Create a dictionary to store one sample text for each intent**

In [17]:
# Build a dictionary with one sample text for each intent
intent_responses = {}

for example in dataset['train']:
    intent_id = example['intent']
    intent_name = label_names[intent_id]

    # Only keep the first example per intent
    if intent_name not in intent_responses:
        intent_responses[intent_name] = example['text']


**Display a sample text for the first five intents**

In [18]:
for intent, example in list(intent_responses.items())[:5]:
    print(f"{intent}: {example}")


direct_deposit: what are the steps for setting up direct deposit for my paycheck
carry_on: if i fly american to los angeles, how many carry ons am i allowed
whisper_mode: change to whisper mode
text: text audrey and tell her i will be there soon
recipe: i want a good recipe that shows me how to bake chocolate chip cookies from scratch


**Define a function to preprocess user input and get the appropriate response from the model**

In [19]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Function to preprocess user input before prediction
def preprocess_input(user_input):
    return preprocess(user_input)  # same as training: lowercase, remove symbols, lemmatize, etc.

def get_response(user_input, model, vectorizer, label_names, intent_responses, threshold=0.3):
    # Preprocess input
    processed = preprocess_input(user_input)
    vec = vectorizer.transform([processed])

    # Predict
    pred = model.predict(vec)
    if model == mlp:
        intent_id = pred.argmax(axis=1)[0]
    else:
        intent_id = pred[0]

    intent = label_names[intent_id]  # <-- this will now give "weather", "greeting", etc.

    # Confidence
    if hasattr(model, "predict_proba"):
        confidence = np.max(model.predict_proba(vec))
    else:
        confidence = 1.0

    if confidence < threshold:
        # Calculate similarity to find best matching intent
        similarities = cosine_similarity(vec, vectorizer.transform(list(intent_responses.values())))
        best_intent_index = similarities.argmax()
        best_intent = list(intent_responses.keys())[best_intent_index]
        best_match = intent_responses[best_intent] # Get the example text for the best intent

        return f"(🤖 unsure — using similarity)\nIntent: {best_intent}\nResponse: {best_match}"
    else:
        response = intent_responses.get(intent, "I'm not sure how to respond to that.")
        return f"(Intent: {intent})\n{response}" # Use intent name here


**Define a chatbot function to interact with the user and provide responses based on the model**

In [20]:
def chat():
    print("🤖 Chatbot: Hello! Ask me anything (type 'exit' to quit).")
    while True:
        user_input = input("You: ")
        if user_input.lower() in ['exit', 'quit']:
            print("🤖 Chatbot: Bye!")
            break

        response = get_response(
            user_input,
            model=lr,  # or nb / rf / mlp
            vectorizer=vectorizer,
            label_names=label_names, # Pass label_encoder.classes_ instead
            intent_responses=intent_responses,
            threshold=0.3
        )
        print(response)

In [21]:
chat()

🤖 Chatbot: Hello! Ask me anything (type 'exit' to quit).
You: exit
🤖 Chatbot: Bye!


**Define simple rule-based responses for each intent and implement the chatbot response function**

In [22]:
# Define simple rule-based responses for each intent
intent_responses = {
    'greeting': "Hello! How can I help you today?",
    'goodbye': "Goodbye! Have a great day!",
    'restaurant_search': "I can help you find a restaurant. What kind of food do you like?",
    'weather': "Sure, let me check the weather for you.",
    'order_status': "Please provide your order ID to check the status.",
    # Add more intent-response mappings as needed
}

# Define chatbot response function
def chatbot_response(user_input):
    # Preprocess the input
    clean_input = preprocess(user_input)
    input_vector = vectorizer.transform([clean_input])

    # Predict the intent using the trained Logistic Regression model
    predicted_label = lr.predict(input_vector)[0]
    intent_name = label_encoder.inverse_transform([predicted_label])[0]

    # Return a relevant response based on the predicted intent
    response = intent_responses.get(
        intent_name, f"I'm not sure how to help with that. (Predicted intent: {intent_name})"
    )
    return response


***Evaluate the Logistic Regression model on the test set and print classification metrics and accuracy***

In [23]:
from sklearn.metrics import classification_report, accuracy_score

# Predict on the test set
y_pred = lr.predict(X_test)

# Convert numeric labels back to text labels
y_test_labels = label_encoder.inverse_transform(y_test)
y_pred_labels = label_encoder.inverse_transform(y_pred)

# Print detailed classification metrics
print("Classification Report:\n")
print(classification_report(y_test_labels, y_pred_labels))

# Print overall accuracy
print("Accuracy:", accuracy_score(y_test_labels, y_pred_labels))


Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        15
           1       1.00      1.00      1.00        13
           2       0.75      1.00      0.86         3
           3       0.82      0.78      0.80        18
           4       0.53      0.83      0.65        12
           5       0.67      0.33      0.44         6
           6       0.25      1.00      0.40         1
           7       1.00      1.00      1.00         5
           8       0.92      0.96      0.94        25
           9       0.73      0.92      0.81        12
          10       0.92      0.89      0.91        27
          11       0.66      0.84      0.74        25
          12       1.00      0.62      0.77         8
          13       0.93      0.87      0.90        30
          14       1.00      0.83      0.91        12
          15       1.00      0.77      0.87        13
          16       0.86      0.79      0.83        24
   

***Build a Streamlit chatbot application with speech recognition and text classification using a pre-trained DistilBERT model***

In [27]:
# chatbot_app.py

import streamlit as st
import speech_recognition as sr
import torch
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import pandas as pd

# Load the CLINC Out-of-Scope dataset
dataset = load_dataset("clinc_oos", "imbalanced")

# Extract training and validation data
train_data = dataset['train']
val_data = dataset['test']  # Using 'test' as validation set in this case

# Prepare the data for training
train_texts = train_data['text']
train_labels = train_data['intent']  # Use 'intent' column for labels
val_texts = val_data['text']
val_labels = val_data['intent']  # Use 'intent' column for labels

# Initialize tokenizer
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

# Tokenize datasets
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=512)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=512)

# Prepare dataset for training
class QADataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {key: torch.tensor(val[idx]) for key, val in self.encodings.items()} | {'labels': torch.tensor(self.labels[idx])}

# Convert data into Dataset objects for PyTorch
train_dataset = QADataset(train_encodings, train_labels)
val_dataset = QADataset(val_encodings, val_labels)

# Load pre-trained model and set up for classification
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=len(set(train_labels)))

# Training setup (You can uncomment to actually train the model)
# from transformers import Trainer, TrainingArguments

# training_args = TrainingArguments(
#     output_dir='./results',
#     num_train_epochs=3,
#     per_device_train_batch_size=8,
#     per_device_eval_batch_size=8,
#     warmup_steps=500,
#     weight_decay=0.01,
#     logging_dir='./logs',
#     logging_steps=10,
#     evaluation_strategy="epoch"
# )

# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_dataset,
#     eval_dataset=val_dataset
# )

# Uncomment to train
# trainer.train()

# Streamlit interface for chatbot
st.title("Chatbot for Classifying Out-of-Scope Questions 🤖")
st.write("Ask a question and the system will classify whether it's in scope or out of scope.")

# Speech-to-text setup
recognizer = sr.Recognizer()
use_mic = st.checkbox("Use Microphone")

if use_mic:
    with sr.Microphone() as source:
        st.write("Say something...")
        audio = recognizer.listen(source)
        try:
            user_input = recognizer.recognize_google(audio)
            st.write(f"You said: {user_input}")
        except sr.UnknownValueError:
            st.error("Could not understand audio.")
            user_input = ""
else:
    user_input = st.text_input("Type your question:")

# Get answer on button press
if st.button("Get Answer") and user_input:
    # Tokenize the input question
    inputs = tokenizer(user_input, return_tensors="pt", truncation=True, padding=True, max_length=512)

    # Get prediction
    with torch.no_grad():
        outputs = model(**inputs)
        predicted_label = torch.argmax(outputs.logits, dim=1).item()

    # Get corresponding answer
    response = train_data['intent'][predicted_label]  # Getting 'intent' from the train data
    st.success(f"Prediction: {response}")


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
2025-04-07 12:46:17.571 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-07 12:46:17.723 
  command:

    streamlit run /usr/local/lib/python3.11/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2025-04-07 12:46:17.724 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-07 12:46:17.726 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-07 12:46:17.728 Thread 'MainThread': missing ScriptRunContext! T